# Validação de Dados

Neste notebook utilizaremos a camada BRONZE para validar informações e encontrar possíveis incossistências na base. Posteriormente realizaremos o tratamento das informações que irão formar a camada silver

In [1]:
# bibliotecas

from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib as plt

In [2]:
# client do bigquery

client = bigquery.Client.from_service_account_json("../credenciais/chave.json")

# Carregando as bases

In [3]:
# base de clientes a partir da camada bronze

query_clientes = """
select
    *
from
  bronze.clientes
"""

df_clientes = client.query(query_clientes).to_dataframe()

c:\Users\Ricardo Wolf\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [4]:
# base de clientes a partir da camada bronze

query_planos = """
select
    *
from
  bronze.clientes
"""

df_planos = client.query(query_planos).to_dataframe()

c:\Users\Ricardo Wolf\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


# Tipagem de Dados

In [5]:
df_clientes.dtypes

ID                                object
api_key                           object
CPF                               object
UF                                object
ID_PLANO                          object
Periodicidade                     object
inicio_data                       object
cancto_data                       object
tipo_cancelamento                 object
status                            object
cupons_total                      object
carregado_em         datetime64[us, UTC]
dtype: object

In [6]:
# adicionando os tipos corretos

df_clientes ['inicio_data'] = pd.to_datetime(df_clientes['inicio_data'])
df_clientes ['cancto_data'] = pd.to_datetime(df_clientes['cancto_data'])

# Inicio da Validação

1. A coluna de api key é única? Imagino que deva ser pois cada cliente recebe uma chave API para acesso ao plano.

In [7]:
# info

df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   ID                 1000 non-null   object             
 1   api_key            1000 non-null   object             
 2   CPF                1000 non-null   object             
 3   UF                 1000 non-null   object             
 4   ID_PLANO           1000 non-null   object             
 5   Periodicidade      1000 non-null   object             
 6   inicio_data        1000 non-null   datetime64[ns]     
 7   cancto_data        572 non-null    datetime64[ns]     
 8   tipo_cancelamento  572 non-null    object             
 9   status             1000 non-null   object             
 10  cupons_total       1000 non-null   object             
 11  carregado_em       1000 non-null   datetime64[us, UTC]
dtypes: datetime64[ns](2), datetime64[us, UTC](1), obj

In [8]:
# count distinct 

quantidade_registros = df_clientes ['ID'].nunique()
quantidade_apis = df_clientes ['api_key'].nunique()

print("Registros:", quantidade_registros)
print("API's:", quantidade_apis)

Registros: 1000
API's: 1000


In [9]:
df_clientes.head()

,ID,api_key,CPF,UF,ID_PLANO,Periodicidade,inicio_data,cancto_data,tipo_cancelamento,status,cupons_total,carregado_em
0,737,ak_000737_88c3e,55877169472,BA,1,Anual,2022-03-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00
1,209,ak_000209_302a3,19467232487,RS,1,Trimestral,2023-10-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00
2,442,ak_000442_70b92,23105927452,SP,1,Anual,2021-05-09,NaT,None,Ativo,169.04,2026-10-03 20:48:15.387499+00:00
3,615,ak_000615_91152,2727857608,MG,1,Anual,2022-09-13,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00
4,621,ak_000621_90dcd,58842524105,RJ,1,Semestral,2025-09-05,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00


# Quantos CPF's distintos temos? Todos tem a quantidade correta de caracteres ?

In [10]:
# criando uma coluna na base para contar caracteres da coluna CPF
df_clientes ['tamanho_cpf'] = df_clientes ['CPF'].astype(str).str.len()

In [11]:
# quantidade de cpf totais
quantidade_cpf = df_clientes ['CPF'].nunique()

# quantidade de cpf com o tamanho correto (11 digitos)
quantidade_cpf_corretos = df_clientes.query("tamanho_cpf == 11")['CPF'].nunique()

print("Total de CPF:", quantidade_cpf)
print("Total de CPFs corretos:", quantidade_cpf_corretos)

Total de CPF: 750
Total de CPFs corretos: 670


Temos 80 CPF's com menos digitos do que o normal. Isso representa 10% do total de CPFs da base.

In [12]:
df_clientes.head()

,ID,api_key,CPF,UF,ID_PLANO,Periodicidade,inicio_data,cancto_data,tipo_cancelamento,status,cupons_total,carregado_em,tamanho_cpf
0,737,ak_000737_88c3e,55877169472,BA,1,Anual,2022-03-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11
1,209,ak_000209_302a3,19467232487,RS,1,Trimestral,2023-10-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11
2,442,ak_000442_70b92,23105927452,SP,1,Anual,2021-05-09,NaT,None,Ativo,169.04,2026-10-03 20:48:15.387499+00:00,11
3,615,ak_000615_91152,2727857608,MG,1,Anual,2022-09-13,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,10
4,621,ak_000621_90dcd,58842524105,RJ,1,Semestral,2025-09-05,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11


In [13]:
df_clientes ['tamanho_cpf'].value_counts()

tamanho_cpf
11    896
10    100
9       4
Name: count, dtype: int64

In [14]:
# coloca zeros a esquerda até fechar 11 caracteres

df_clientes["cpf_11"] = df_clientes["CPF"].astype(str).str.zfill(11)

In [15]:
df_clientes.head()

,ID,api_key,CPF,UF,ID_PLANO,Periodicidade,inicio_data,cancto_data,tipo_cancelamento,status,cupons_total,carregado_em,tamanho_cpf,cpf_11
0,737,ak_000737_88c3e,55877169472,BA,1,Anual,2022-03-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,55877169472
1,209,ak_000209_302a3,19467232487,RS,1,Trimestral,2023-10-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,19467232487
2,442,ak_000442_70b92,23105927452,SP,1,Anual,2021-05-09,NaT,None,Ativo,169.04,2026-10-03 20:48:15.387499+00:00,11,23105927452
3,615,ak_000615_91152,2727857608,MG,1,Anual,2022-09-13,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,10,02727857608
4,621,ak_000621_90dcd,58842524105,RJ,1,Semestral,2025-09-05,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,58842524105


# Existe linhas duplicadas?

In [16]:
df_clientes.duplicated().sum()      
df_clientes[df_clientes.duplicated(keep=False)]   

,ID,api_key,CPF,UF,ID_PLANO,Periodicidade,inicio_data,cancto_data,tipo_cancelamento,status,cupons_total,carregado_em,tamanho_cpf,cpf_11


# Como são nossos dados nulos?

In [17]:
df_clientes.isna().sum()

ID                     0
api_key                0
CPF                    0
UF                     0
ID_PLANO               0
Periodicidade          0
inicio_data            0
cancto_data          428
tipo_cancelamento    428
status                 0
cupons_total           0
carregado_em           0
tamanho_cpf            0
cpf_11                 0
dtype: int64

Não temos problemas com dados nulos, pois as colunas cancto_data e tipo_cancelamento já eram esperadas

# Inicio Data é maior que o dia atual?

In [19]:
hoje = pd.Timestamp.today().normalize()   
validacao_data_inicio = df_clientes.query("inicio_data > @hoje")
len(validacao_data_inicio)

0

# A data do cancelamento é maior que a data de inicio?

In [20]:
df_validacao_data_cancelamento = df_clientes.query("cancto_data < inicio_data")
df_validacao_data_cancelamento.head()

,ID,api_key,CPF,UF,ID_PLANO,Periodicidade,inicio_data,cancto_data,tipo_cancelamento,status,cupons_total,carregado_em,tamanho_cpf,cpf_11


# Existe tipo de cancelamento sem data de cancelamento?

In [21]:
# motivo preenchido, mas sem data de cancelamento
motivo_sem_data = df_clientes[df_clientes["tipo_cancelamento"].notna() & df_clientes["cancto_data"].isna()]

# data de cancelamento preenchida, mas sem motivo
data_sem_motivo = df_clientes[df_clientes["cancto_data"].notna() & df_clientes["tipo_cancelamento"].isna()]

len(motivo_sem_data), len(data_sem_motivo)

(0, 0)

In [22]:
df_clientes.head()

,ID,api_key,CPF,UF,ID_PLANO,Periodicidade,inicio_data,cancto_data,tipo_cancelamento,status,cupons_total,carregado_em,tamanho_cpf,cpf_11
0,737,ak_000737_88c3e,55877169472,BA,1,Anual,2022-03-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,55877169472
1,209,ak_000209_302a3,19467232487,RS,1,Trimestral,2023-10-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,19467232487
2,442,ak_000442_70b92,23105927452,SP,1,Anual,2021-05-09,NaT,None,Ativo,169.04,2026-10-03 20:48:15.387499+00:00,11,23105927452
3,615,ak_000615_91152,2727857608,MG,1,Anual,2022-09-13,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,10,02727857608
4,621,ak_000621_90dcd,58842524105,RJ,1,Semestral,2025-09-05,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,58842524105


# Quais os motivos do cancelamento?

In [23]:
df_clientes["tipo_cancelamento"].value_counts(dropna=False)

tipo_cancelamento
None            428
Concorrentes    186
Produto         110
Preço           103
Migração         87
Outro            86
Name: count, dtype: int64

# O mesmo plano está ativo simultaneamente no mesmo CPF ?

In [24]:
df_clientes['status'].value_counts()

status
Cancelado       572
Ativo           363
Inadimplente     65
Name: count, dtype: int64

In [25]:
# somente ativos

df_ativos = df_clientes.query("status != 'Cancelado'")
df_ativos.head()

,ID,api_key,CPF,UF,ID_PLANO,Periodicidade,inicio_data,cancto_data,tipo_cancelamento,status,cupons_total,carregado_em,tamanho_cpf,cpf_11
0,737,ak_000737_88c3e,55877169472,BA,1,Anual,2022-03-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,55877169472
1,209,ak_000209_302a3,19467232487,RS,1,Trimestral,2023-10-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,19467232487
2,442,ak_000442_70b92,23105927452,SP,1,Anual,2021-05-09,NaT,None,Ativo,169.04,2026-10-03 20:48:15.387499+00:00,11,23105927452
3,615,ak_000615_91152,2727857608,MG,1,Anual,2022-09-13,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,10,02727857608
4,621,ak_000621_90dcd,58842524105,RJ,1,Semestral,2025-09-05,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,58842524105


In [26]:
df_ativos_validacao = df_ativos.groupby(['cpf_11', 'ID_PLANO']).size().reset_index(name='contagem_contratos')
df_ativos_validacao['contagem_contratos'].value_counts()

contagem_contratos
1    428
Name: count, dtype: int64

In [ ]:
df_clientes.head()

,ID,api_key,CPF,UF,ID_PLANO,Periodicidade,inicio_data,cancto_data,tipo_cancelamento,status,cupons_total,carregado_em,tamanho_cpf,cpf_11
0,737,ak_000737_88c3e,55877169472,BA,1,Anual,2022-03-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,55877169472
1,209,ak_000209_302a3,19467232487,RS,1,Trimestral,2023-10-12,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,19467232487
2,442,ak_000442_70b92,23105927452,SP,1,Anual,2021-05-09,NaT,None,Ativo,169.04,2026-10-03 20:48:15.387499+00:00,11,23105927452
3,615,ak_000615_91152,2727857608,MG,1,Anual,2022-09-13,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,10,02727857608
4,621,ak_000621_90dcd,58842524105,RJ,1,Semestral,2025-09-05,NaT,None,Ativo,0.0,2026-10-03 20:48:15.387499+00:00,11,58842524105
